## 1:Load Data and save to paraquet

In [1]:
# 1load data code
import pandas as pd
import numpy as np
import os
import joblib
from joblib import Parallel, delayed
from tqdm import tqdm
import warnings

warnings.filterwarnings("ignore")

print("Library versions:")
print(f"  pandas  : {pd.__version__}")
print(f"  numpy   : {np.__version__}")
print(f"  joblib  : {joblib.__version__}")
print()

# I used datasets given on "https://physionet.org/content/challenge-2019/1.0.0/" and combined both datasets
DIR_A       = "/kaggle/input/datasets/ibrahimfiftysix/physionet/training_setA/training_setA"
DIR_B       = "/kaggle/input/datasets/ibrahimfiftysix/physionet/training_setB/training_setB"
OUT_DIR     = "/kaggle/working/"
RANDOM_SEED = 42
N_JOBS      = -1

#will load a single file(psv) and tag with a patint id
def load_patient_file(filepath):
    try:
        df = pd.read_csv(filepath, sep="|")
        patient_id = os.path.splitext(os.path.basename(filepath))[0]
        df["patient_id"] = patient_id
        # Determine set A or B from the parent directory name
        parent = os.path.basename(os.path.dirname(filepath))
        df["set"] = "A" if parent.endswith("A") else "B"
        return df
    except Exception as e:
        print(f"failed to load {filepath}: {e}")
        return None

#collect all files
files_a = sorted([os.path.join(DIR_A, f) for f in os.listdir(DIR_A) if f.endswith(".psv")])
files_b = sorted([os.path.join(DIR_B, f) for f in os.listdir(DIR_B) if f.endswith(".psv")])
all_files = files_a + files_b

print(f"Files found  →  Set A: {len(files_a):,}  |  Set B: {len(files_b):,}  |  Total: {len(all_files):,}")
print()

#since there are 40000+ files using parallel loading
print("started loading patient files")
results = Parallel(n_jobs=N_JOBS, backend="loky")(
    delayed(load_patient_file)(fp) for fp in tqdm(all_files, desc="Patients", unit="file")
)

#failed files
loaded   = [r for r in results if r is not None]
n_failed = len(results) - len(loaded)

print(f"\nSuccessfully loaded : {len(loaded):,}  |  Failed: {n_failed}")
print()

df_raw = pd.concat(loaded, ignore_index=True)

#save to parquet
out_path = os.path.join(OUT_DIR, "raw_combined.parquet")
df_raw.to_parquet(out_path, index=False)
print(f"Saved : {out_path}")
print()

# confirm by printing details
print("═" * 60)
print("df_raw.head(10)")
print("═" * 60)
display(df_raw.head(10))
print()

print("═" * 60)
print("df_raw.info()")
print("═" * 60)
df_raw.info()
print()


total_patients = df_raw["patient_id"].nunique()
total_rows = len(df_raw)
sepsis_patients = df_raw.groupby("patient_id")["SepsisLabel"].max()
n_sepsis_patients = int(sepsis_patients.sum())
pct_sepsis_patients = n_sepsis_patients / total_patients * 100
n_sepsis_rows = int(df_raw["SepsisLabel"].sum())
pct_sepsis_rows = n_sepsis_rows / total_rows * 100
mem_mb  = df_raw.memory_usage(deep=True).sum() / 1e6

print("═" * 60)
print("DATA SUMMARY")
print("═" * 60)
print(f"  Total patients:          {total_patients:,}")
print(f"  Total rows:              {total_rows:,}")
print(f"  Sepsis-positive patients:{n_sepsis_patients:,} ({pct_sepsis_patients:.1f}%)")
print(f"  Sepsis-positive rows:    {n_sepsis_rows:,} ({pct_sepsis_rows:.1f}%)")
print(f"  Memory usage:            {mem_mb:.1f} MB")
print(f"  Failed files:            {n_failed}")
print("═" * 60)


Library versions:
  pandas  : 2.3.3
  numpy   : 2.4.6
  joblib  : 1.5.3

Files found  →  Set A: 20,322  |  Set B: 19,102  |  Total: 39,424

started loading patient files


Patients: 100%|██████████| 39424/39424 [01:20<00:00, 488.08file/s]



Successfully loaded : 39,424  |  Failed: 0

Saved : /kaggle/working/raw_combined.parquet

════════════════════════════════════════════════════════════
df_raw.head(10)
════════════════════════════════════════════════════════════


,HR,O2Sat,Temp,SBP,MAP,DBP,Resp,EtCO2,BaseExcess,HCO3,...,Platelets,Age,Gender,Unit1,Unit2,HospAdmTime,ICULOS,SepsisLabel,patient_id,set
0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,83.14,0,NaN,NaN,-0.03,1,0,p000001,A
1,97.0,95.0,NaN,98.0,75.33,NaN,19.0,NaN,NaN,NaN,...,NaN,83.14,0,NaN,NaN,-0.03,2,0,p000001,A
2,89.0,99.0,NaN,122.0,86.00,NaN,22.0,NaN,NaN,NaN,...,NaN,83.14,0,NaN,NaN,-0.03,3,0,p000001,A
3,90.0,95.0,NaN,NaN,NaN,NaN,30.0,NaN,24.0,NaN,...,NaN,83.14,0,NaN,NaN,-0.03,4,0,p000001,A
4,103.0,88.5,NaN,122.0,91.33,NaN,24.5,NaN,NaN,NaN,...,NaN,83.14,0,NaN,NaN,-0.03,5,0,p000001,A
5,110.0,91.0,NaN,NaN,NaN,NaN,22.0,NaN,NaN,NaN,...,NaN,83.14,0,NaN,NaN,-0.03,6,0,p000001,A
6,108.0,92.0,36.11,123.0,77.00,NaN,29.0,NaN,NaN,NaN,...,NaN,83.14,0,NaN,NaN,-0.03,7,0,p000001,A
7,106.0,90.5,NaN,93.0,76.33,NaN,29.0,NaN,NaN,NaN,...,NaN,83.14,0,NaN,NaN,-0.03,8,0,p000001,A
8,104.0,95.0,NaN,133.0,88.33,NaN,26.0,NaN,NaN,NaN,...,NaN,83.14,0,NaN,NaN,-0.03,9,0,p000001,A
9,102.0,91.0,NaN,134.0,87.33,NaN,30.0,NaN,NaN,NaN,...,NaN,83.14,0,NaN,NaN,-0.03,10,0,p000001,A



════════════════════════════════════════════════════════════
df_raw.info()
════════════════════════════════════════════════════════════
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1517717 entries, 0 to 1517716
Data columns (total 43 columns):
 #   Column            Non-Null Count    Dtype  
---  ------            --------------    -----  
 0   HR                1368210 non-null  float64
 1   O2Sat             1319562 non-null  float64
 2   Temp              513178 non-null   float64
 3   SBP               1295988 non-null  float64
 4   MAP               1329201 non-null  float64
 5   DBP               1035893 non-null  float64
 6   Resp              1286463 non-null  float64
 7   EtCO2             55093 non-null    float64
 8   BaseExcess        84062 non-null    float64
 9   HCO3              64946 non-null    float64
 10  FiO2              128514 non-null   float64
 11  pH                106769 non-null   float64
 12  PaCO2             85497 non-null    float64
 13  SaO2      

## 2. Data analysis eda

In [2]:
# 2 data analysis code
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import pandas as pd

sns.set_theme(style="darkgrid", palette="deep")
RANDOM_SEED = 42
OUT_DIR = "/kaggle/working/"


feature_cols = [c for c in df_raw.columns if c not in ["patient_id", "set"]]
feature_cols = [c for c in feature_cols if c != "SepsisLabel"]

#1 missingness table

sepsis_mask = df_raw["SepsisLabel"] == 1

pct_all = df_raw[feature_cols].isnull().mean() * 100
pct_sep = df_raw.loc[sepsis_mask, feature_cols].isnull().mean() * 100
pct_non = df_raw.loc[~sepsis_mask, feature_cols].isnull().mean() * 100

df_missing = pd.DataFrame({
    "feature": feature_cols,
    "pct_missing_overall": pct_all.values,
    "pct_missing_sepsis": pct_sep.values,
    "pct_missing_nonsepsis": pct_non.values
}).sort_values("pct_missing_overall", ascending=False).reset_index(drop=True)

print("missingness table (sorted by overall %)")
display(df_missing)
df_missing.to_csv(OUT_DIR + "missingness_table.csv", index=False)

#heatmap of missingness
fig, ax = plt.subplots(figsize=(14, 10))
heatmap_data = df_missing.set_index("feature")[
    ["pct_missing_overall", "pct_missing_sepsis", "pct_missing_nonsepsis"]
]
sns.heatmap(
    heatmap_data, annot=True, fmt=".1f", cmap="YlOrRd",
    linewidths=0.5, ax=ax, cbar_kws={"label": "% missing"}
)
ax.set_title("missingness % per feature", fontsize=14)
ax.set_xlabel("")
ax.set_ylabel("")
plt.tight_layout()
plt.savefig(OUT_DIR + "REPORT_missingness_heatmap.png", dpi=150, bbox_inches="tight")
plt.show()

# 2 class imbalance

n_pos_rows = int(sepsis_mask.sum())
n_neg_rows = int((~sepsis_mask).sum())
n_pos_patients = int(df_raw.groupby("patient_id")["SepsisLabel"].max().sum())
n_neg_patients = df_raw["patient_id"].nunique() - n_pos_patients

print(f"\nsepsis-positive patients: {n_pos_patients:,}")
print(f"sepsis-negative patients: {n_neg_patients:,}")
print(f"positive rows: {n_pos_rows:,} ({n_pos_rows / len(df_raw) * 100:.2f}%)")

fig, ax = plt.subplots(figsize=(8, 5))
bars = ax.bar(
    ["non-sepsis (0)", "sepsis (1)"],
    [n_neg_rows, n_pos_rows],
    color=["#4c72b0", "#dd8452"],
    edgecolor="black", linewidth=0.5
)
for bar, val in zip(bars, [n_neg_rows, n_pos_rows]):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 1000,
            f"{val:,}", ha="center", va="bottom", fontsize=11)
ax.set_title("row-level class distribution", fontsize=14)
ax.set_ylabel("number of rows")
plt.tight_layout()
plt.savefig(OUT_DIR + "REPORT_class_imbalance.png", dpi=150, bbox_inches="tight")
plt.show()

# 3 icu stay length


stay_lengths = df_raw.groupby("patient_id").size()

fig, ax = plt.subplots(figsize=(8, 5))
ax.hist(stay_lengths, bins=80, color="#4c72b0", edgecolor="black", linewidth=0.3)
ax.axvline(stay_lengths.median(), color="red", linestyle="--", label=f"median = {stay_lengths.median():.0f}h")
ax.set_title("icu stay length distribution", fontsize=14)
ax.set_xlabel("hours in icu")
ax.set_ylabel("number of patients")
ax.legend()
plt.tight_layout()
plt.savefig(OUT_DIR + "REPORT_icu_stay_distribution.png", dpi=150, bbox_inches="tight")
plt.show()

# 4 sepsis onset timing


# for each sepsis patient find the first hour where label flips to 1
sepsis_patients = df_raw[df_raw["SepsisLabel"] == 1]
onset_times = sepsis_patients.groupby("patient_id")["ICULOS"].min()

fig, ax = plt.subplots(figsize=(8, 5))
ax.hist(onset_times, bins=80, color="#dd8452", edgecolor="black", linewidth=0.3)
ax.axvline(onset_times.median(), color="red", linestyle="--", label=f"median = {onset_times.median():.0f}h")
ax.set_title("sepsis onset timing (first SepsisLabel=1)", fontsize=14)
ax.set_xlabel("ICULOS (hours after icu admission)")
ax.set_ylabel("number of patients")
ax.legend()
plt.tight_layout()
plt.savefig(OUT_DIR + "REPORT_sepsis_onset_timing.png", dpi=150, bbox_inches="tight")
plt.show()

# 5 vital signs over time for 5 random sepsis patients

sepsis_ids = df_raw[df_raw["SepsisLabel"] == 1]["patient_id"].unique()
rng = np.random.RandomState(RANDOM_SEED)
sample_ids = rng.choice(sepsis_ids, size=5, replace=False)

vitals = ["HR", "O2Sat", "MAP", "Resp"]
colors = ["#4c72b0", "#55a868", "#c44e52", "#8172b3"]

fig, axes = plt.subplots(5, 1, figsize=(18, 15), sharex=False)

for i, pid in enumerate(sample_ids):
    ax = axes[i]
    pat = df_raw[df_raw["patient_id"] == pid].sort_values("ICULOS")
    onset_hour = pat.loc[pat["SepsisLabel"] == 1, "ICULOS"].min()

    for vital, color in zip(vitals, colors):
        ax.plot(pat["ICULOS"], pat[vital], label=vital, color=color, linewidth=1.2)

    ax.axvline(onset_hour, color="red", linestyle="--", linewidth=1.5, label="sepsis onset")
    ax.set_title(f"patient {pid}", fontsize=11)
    ax.set_ylabel("value")
    ax.legend(loc="upper right", fontsize=8, ncol=5)

axes[-1].set_xlabel("ICULOS (hours)")
plt.suptitle("vital signs over time - 5 sepsis patients", fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig(OUT_DIR + "REPORT_vitals_over_time.png", dpi=150, bbox_inches="tight")
plt.show()

# 6 feature correlation matrix

corr = df_raw[feature_cols].corr(method="pearson")

fig, ax = plt.subplots(figsize=(18, 16))
sns.heatmap(
    corr, cmap="RdBu_r", center=0, vmin=-1, vmax=1,
    linewidths=0.2, ax=ax, square=True,
    cbar_kws={"shrink": 0.8, "label": "pearson r"},
    xticklabels=True, yticklabels=True
)
ax.set_title("feature correlation matrix", fontsize=14)
ax.tick_params(axis="both", labelsize=7)
plt.tight_layout()
plt.savefig(OUT_DIR + "REPORT_feature_correlation.png", dpi=150, bbox_inches="tight")
plt.show()

# summary

most_missing = df_missing.iloc[0]
least_missing = df_missing.iloc[-1]
median_stay = stay_lengths.median()
median_onset = onset_times.median()

print("\n" + "=" * 60)
print("eda complete - key findings")
print("=" * 60)
print(f"  most missing feature:  {most_missing['feature']} ({most_missing['pct_missing_overall']:.1f}% missing)")
print(f"  least missing feature: {least_missing['feature']}")
print(f"  sepsis patients: {n_pos_patients:,} | non-sepsis: {n_neg_patients:,}")
print(f"  median icu stay: {median_stay:.0f} hours")
print(f"  median sepsis onset: {median_onset:.0f} hours after icu admit")
print("=" * 60)


missingness table (sorted by overall %)


,feature,pct_missing_overall,pct_missing_sepsis,pct_missing_nonsepsis
0,Bilirubin_direct,99.808199,99.544278,99.813056
1,Fibrinogen,99.338941,98.822414,99.348448
2,TroponinI,99.065570,98.898976,99.068636
3,Bilirubin_total,98.515731,97.134420,98.541154
4,Alkalinephos,98.397066,96.784425,98.426747
5,AST,98.381055,96.769842,98.410710
6,Lactate,97.312872,92.974589,97.392719
7,PTT,97.012091,95.592256,97.038223
8,SaO2,96.513579,93.517810,96.568717
9,EtCO2,96.370008,89.470998,96.496986



sepsis-positive patients: 2,881
sepsis-negative patients: 36,543
positive rows: 27,429 (1.81%)

eda complete - key findings
  most missing feature:  Bilirubin_direct (99.8% missing)
  least missing feature: ICULOS
  sepsis patients: 2,881 | non-sepsis: 36,543
  median icu stay: 39 hours
  median sepsis onset: 29 hours after icu admit


## 3.train/val/test spilt

In [3]:
# cell 3 train/val/test split


#split at PATIENT level all rows from one patient stay together
#stratify by sepsis status so each split has similar sepsis ratio
#never shuffled rows within a patient, preserve time order

import pandas as pd
import numpy as np
import json
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split

RANDOM_SEED = 42
OUT_DIR = "/kaggle/working/"

# 1 patient-level summary

df_patients = df_raw.groupby("patient_id").agg(
    set_label=("set", "first"),
    is_sepsis_patient=("SepsisLabel", "max"),
    n_rows=("SepsisLabel", "size"),
    max_iculos=("ICULOS", "max")
).reset_index()

df_patients["is_sepsis_patient"] = df_patients["is_sepsis_patient"].astype(int)

print("patient-level summary")
display(df_patients.head(10))

# 2 stratified split 70 train, 15 val, 15 test

#70 train 30 temp
train_ids, temp_ids = train_test_split(
    df_patients["patient_id"],
    test_size=0.30,
    stratify=df_patients["is_sepsis_patient"],
    random_state=RANDOM_SEED
)
#second spliton temp 15 15
temp_df = df_patients[df_patients["patient_id"].isin(temp_ids)]
val_ids, test_ids = train_test_split(
    temp_df["patient_id"],
    test_size=0.50,
    stratify=temp_df["is_sepsis_patient"],
    random_state=RANDOM_SEED
)

# 3 filter df_raw and sort by patient then time

train_set = set(train_ids)
val_set = set(val_ids)
test_set = set(test_ids)

df_train = df_raw[df_raw["patient_id"].isin(train_set)].sort_values(["patient_id", "ICULOS"]).reset_index(drop=True)
df_val   = df_raw[df_raw["patient_id"].isin(val_set)].sort_values(["patient_id", "ICULOS"]).reset_index(drop=True)
df_test  = df_raw[df_raw["patient_id"].isin(test_set)].sort_values(["patient_id", "ICULOS"]).reset_index(drop=True)

# 4 save parquet files and split ids json

df_train.to_parquet(OUT_DIR + "train.parquet", index=False)
df_val.to_parquet(OUT_DIR + "val.parquet", index=False)
df_test.to_parquet(OUT_DIR + "test.parquet", index=False)

split_ids = {
    "train": sorted(train_ids.tolist()),
    "val": sorted(val_ids.tolist()),
    "test": sorted(test_ids.tolist())
}
with open(OUT_DIR + "split_ids.json", "w") as f:
    json.dump(split_ids, f)

print(f"saved train.parquet ({len(df_train):,} rows)")
print(f"saved val.parquet ({len(df_val):,} rows)")
print(f"saved test.parquet ({len(df_test):,} rows)")
print(f"saved split_ids.json")

# 5 verify disjoint splits

assert train_set.isdisjoint(val_set), "train/val overlap"
assert train_set.isdisjoint(test_set), "train/test overlap"
assert val_set.isdisjoint(test_set), "val/test overlap"
print("\nsplit complete - no patient appears in more than one split")

# 6 split report

def split_stats(name, df):
    n_patients = df["patient_id"].nunique()
    n_rows = len(df)
    sep_patients = int(df.groupby("patient_id")["SepsisLabel"].max().sum())
    sep_rows = int(df["SepsisLabel"].sum())
    sep_pct = sep_rows / n_rows * 100 if n_rows > 0 else 0
    return [name, f"{n_patients:,}", f"{n_rows:,}", f"{sep_patients:,}", f"{sep_rows:,}", f"{sep_pct:.2f}%"]

report_rows = [
    split_stats("train", df_train),
    split_stats("val", df_val),
    split_stats("test", df_test),
]

col_headers = ["split", "patients", "rows", "sepsis patients", "sepsis rows", "sepsis row %"]

# print text table
print(f"\n{'split':<8} {'patients':>10} {'rows':>12} {'sep patients':>15} {'sep rows':>12} {'sep row %':>10}")
print("-" * 70)
for row in report_rows:
    print(f"{row[0]:<8} {row[1]:>10} {row[2]:>12} {row[3]:>15} {row[4]:>12} {row[5]:>10}")

fig, ax = plt.subplots(figsize=(10, 3))
ax.axis("off")
table = ax.table(
    cellText=report_rows,
    colLabels=col_headers,
    cellLoc="center",
    loc="center"
)
table.auto_set_font_size(False)
table.set_fontsize(10)
table.scale(1.2, 1.6)

# style header row
for j in range(len(col_headers)):
    table[0, j].set_facecolor("#4c72b0")
    table[0, j].set_text_props(color="white", fontweight="bold")

ax.set_title("train / val / test split summary", fontsize=13, pad=20)
plt.tight_layout()
plt.savefig(OUT_DIR + "REPORT_data_split_summary.png", dpi=150, bbox_inches="tight")
plt.show()


patient-level summary


,patient_id,set_label,is_sepsis_patient,n_rows,max_iculos
0,p000001,A,0,54,54
1,p000002,A,0,23,23
2,p000003,A,0,48,48
3,p000004,A,0,29,29
4,p000005,A,0,48,49
5,p000006,A,0,17,19
6,p000007,A,0,45,45
7,p000008,A,0,40,40
8,p000009,A,1,258,258
9,p000010,A,0,23,25


saved train.parquet (1,060,185 rows)
saved val.parquet (228,969 rows)
saved test.parquet (228,563 rows)
saved split_ids.json

split complete - no patient appears in more than one split

split      patients         rows    sep patients     sep rows  sep row %
----------------------------------------------------------------------
train        27,596    1,060,185           2,017       19,205      1.81%
val           5,914      228,969             432        4,107      1.79%
test          5,914      228,563             432        4,117      1.80%


## 4. Missing Values

In [4]:
# cell 4 - missing value imputation

#add binary missingness indicators for clinically important cols
#forward fill within each patient carry last known value forward
#fill remaining nans with training set medians 

import pandas as pd
import numpy as np
import json
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

OUT_DIR = "/kaggle/working/"

try:
    _ = df_train.shape
except NameError:
    df_train = pd.read_parquet(OUT_DIR + "train.parquet")
    df_val   = pd.read_parquet(OUT_DIR + "val.parquet")
    df_test  = pd.read_parquet(OUT_DIR + "test.parquet")
    print("loaded splits from parquet")

#the 40 feature columns
feature_cols = [
    "HR", "O2Sat", "Temp", "SBP", "MAP", "DBP", "Resp", "EtCO2",
    "BaseExcess", "HCO3", "FiO2", "pH", "PaCO2", "SaO2", "AST", "BUN",
    "Alkalinephos", "Calcium", "Chloride", "Creatinine", "Bilirubin_direct",
    "Glucose", "Lactate", "Magnesium", "Phosphate", "Potassium",
    "Bilirubin_total", "TroponinI", "Hct", "Hgb", "PTT", "WBC",
    "Fibrinogen", "Platelets", "Age", "Gender", "Unit1", "Unit2",
    "HospAdmTime", "ICULOS"
]

# 1 missingness indicators 


missing_indicator_cols = [
    "Lactate", "WBC", "Creatinine", "Bilirubin_total",
    "Platelets", "PTT", "pH", "Temp", "Resp"
]

for col in missing_indicator_cols:
    flag = f"{col}_was_missing"
    df_train[flag] = df_train[col].isna().astype(int)
    df_val[flag]   = df_val[col].isna().astype(int)
    df_test[flag]  = df_test[col].isna().astype(int)

indicator_names = [f"{c}_was_missing" for c in missing_indicator_cols]
print(f"added {len(indicator_names)} missingness indicator columns")

# for the report chart
before_pct = df_train[feature_cols].isnull().mean() * 100

#forward fill within each patient


#sort by patient then time so ffill goes forward in time
df_train = df_train.sort_values(["patient_id", "ICULOS"]).reset_index(drop=True)
df_val   = df_val.sort_values(["patient_id", "ICULOS"]).reset_index(drop=True)
df_test  = df_test.sort_values(["patient_id", "ICULOS"]).reset_index(drop=True)

df_train[feature_cols] = df_train.groupby("patient_id")[feature_cols].transform(lambda x: x.ffill())
df_val[feature_cols]   = df_val.groupby("patient_id")[feature_cols].transform(lambda x: x.ffill())
df_test[feature_cols]  = df_test.groupby("patient_id")[feature_cols].transform(lambda x: x.ffill())

after_ffill_pct = df_train[feature_cols].isnull().mean() * 100
print(f"after ffill - remaining nan features: {(after_ffill_pct > 0).sum()}")


# median imputation from training set only


#compute medians from train only not val/test
training_medians = df_train[feature_cols].median().to_dict()

df_train[feature_cols] = df_train[feature_cols].fillna(training_medians)
df_val[feature_cols]   = df_val[feature_cols].fillna(training_medians)
df_test[feature_cols]  = df_test[feature_cols].fillna(training_medians)

#save medians for inference later
with open(OUT_DIR + "imputer_medians.json", "w") as f:
    json.dump(training_medians, f, indent=2)

print("saved imputer_medians.json")

#verify


after_pct = df_train[feature_cols].isnull().mean() * 100

print("\nnan counts after imputation:")
for name, df in [("train", df_train), ("val", df_val), ("test", df_test)]:
    total_nans = df[feature_cols].isnull().sum().sum()
    print(f"  {name}: {total_nans} nans remaining")

print()
display(df_train.head(10))

sample_cols = ["HR", "Lactate", "WBC", "Lactate_was_missing", "WBC_was_missing"]
print("\nsample of imputed values with missingness flags:")
display(df_train[sample_cols].head(20))

all_cols_after = list(df_train.columns)
n_indicator = len(indicator_names)
n_total = len([c for c in all_cols_after if c not in ["patient_id", "set", "SepsisLabel"]])
print(f"\nnew missingness indicator columns added: {n_indicator}")
print(f"total feature columns after imputation step: {n_total}")

#save

df_train.to_parquet(OUT_DIR + "train_imputed.parquet", index=False)
df_val.to_parquet(OUT_DIR + "val_imputed.parquet", index=False)
df_test.to_parquet(OUT_DIR + "test_imputed.parquet", index=False)

print(f"\nsaved train_imputed.parquet ({len(df_train):,} rows)")
print(f"saved val_imputed.parquet ({len(df_val):,} rows)")
print(f"saved test_imputed.parquet ({len(df_test):,} rows)")


#report

top15 = before_pct.sort_values(ascending=False).head(15)
top15_names = top15.index.tolist()

fig, ax = plt.subplots(figsize=(14, 6))
x = np.arange(len(top15_names))
w = 0.35

bars_before = ax.bar(x - w/2, before_pct[top15_names], w, label="before imputation", color="#dd8452", edgecolor="black", linewidth=0.3)
bars_after  = ax.bar(x + w/2, after_pct[top15_names], w, label="after imputation", color="#4c72b0", edgecolor="black", linewidth=0.3)

ax.set_xticks(x)
ax.set_xticklabels(top15_names, rotation=45, ha="right", fontsize=9)
ax.set_ylabel("% missing")
ax.set_title("missingness before vs after imputation (top 15 features, train set)", fontsize=13)
ax.legend()

# add value labels on before bars
for bar in bars_before:
    h = bar.get_height()
    if h > 1:
        ax.text(bar.get_x() + bar.get_width()/2, h + 0.5, f"{h:.0f}%", ha="center", va="bottom", fontsize=7)

plt.tight_layout()
plt.savefig(OUT_DIR + "REPORT_imputation_before_after.png", dpi=150, bbox_inches="tight")
plt.show()


added 9 missingness indicator columns
after ffill - remaining nan features: 37
saved imputer_medians.json

nan counts after imputation:
  train: 0 nans remaining
  val: 0 nans remaining
  test: 0 nans remaining



,HR,O2Sat,Temp,SBP,MAP,DBP,Resp,EtCO2,BaseExcess,HCO3,...,set,Lactate_was_missing,WBC_was_missing,Creatinine_was_missing,Bilirubin_total_was_missing,Platelets_was_missing,PTT_was_missing,pH_was_missing,Temp_was_missing,Resp_was_missing
0,83.0,98.0,36.83,121.0,81.00,62.0,18.0,34.0,0.0,24.0,...,A,1,1,1,1,1,1,1,1,1
1,97.0,95.0,36.83,98.0,75.33,62.0,19.0,34.0,0.0,24.0,...,A,1,1,1,1,1,1,1,1,0
2,89.0,99.0,36.83,122.0,86.00,62.0,22.0,34.0,0.0,24.0,...,A,1,1,1,1,1,1,1,1,0
3,90.0,95.0,36.83,122.0,86.00,62.0,30.0,34.0,24.0,24.0,...,A,1,1,1,1,1,1,0,1,0
4,103.0,88.5,36.83,122.0,91.33,62.0,24.5,34.0,24.0,24.0,...,A,1,1,1,1,1,1,1,1,0
5,110.0,91.0,36.83,122.0,91.33,62.0,22.0,34.0,24.0,24.0,...,A,1,1,1,1,1,1,1,1,0
6,108.0,92.0,36.11,123.0,77.00,62.0,29.0,34.0,24.0,24.0,...,A,1,1,1,1,1,1,1,0,0
7,106.0,90.5,36.11,93.0,76.33,62.0,29.0,34.0,24.0,24.0,...,A,1,1,1,1,1,1,1,1,0
8,104.0,95.0,36.11,133.0,88.33,62.0,26.0,34.0,24.0,24.0,...,A,1,1,1,1,1,1,1,1,0
9,102.0,91.0,36.11,134.0,87.33,62.0,30.0,34.0,24.0,24.0,...,A,1,1,1,1,1,1,1,1,0



sample of imputed values with missingness flags:


,HR,Lactate,WBC,Lactate_was_missing,WBC_was_missing
0,83.0,1.59,10.3,1,1
1,97.0,1.59,10.3,1,1
2,89.0,1.59,10.3,1,1
3,90.0,1.59,10.3,1,1
4,103.0,1.59,10.3,1,1
5,110.0,1.59,10.3,1,1
6,108.0,1.59,10.3,1,1
7,106.0,1.59,10.3,1,1
8,104.0,1.59,10.3,1,1
9,102.0,1.59,10.3,1,1



new missingness indicator columns added: 9
total feature columns after imputation step: 49

saved train_imputed.parquet (1,060,185 rows)
saved val_imputed.parquet (228,969 rows)
saved test_imputed.parquet (228,563 rows)


## 5.feature engineering

In [5]:
# cell 5feature engineering

#all rolling/lag ops use groupby('patient_id') and only look backward
#no future data ever leaks into any feature

import pandas as pd
import numpy as np
import json
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

try:
    _ = df_train.shape
except NameError:
    df_train = pd.read_parquet(OUT_DIR + "train_imputed.parquet")
    df_val   = pd.read_parquet(OUT_DIR + "val_imputed.parquet")
    df_test  = pd.read_parquet(OUT_DIR + "test_imputed.parquet")
    print("loaded imputed splits from parquet")

shape_before = df_train.shape
cols_before = [c for c in df_train.columns if c not in ["patient_id", "set", "SepsisLabel"]]


def engineer_features(df):
    df = df.copy()
    df = df.sort_values(["patient_id", "ICULOS"]).reset_index(drop=True)

    # group 1 rolling stats over last 6 hours
    rolling_cols = ["HR", "O2Sat", "MAP", "Resp", "Temp", "WBC",
                    "Lactate", "Creatinine", "SBP", "DBP"]

    for col in rolling_cols:
        g = df.groupby("patient_id")[col]
        df[f"{col}_mean6h"] = g.transform(lambda x: x.rolling(6, min_periods=1).mean())
        df[f"{col}_std6h"]  = g.transform(lambda x: x.rolling(6, min_periods=1).std())
        df[f"{col}_min6h"]  = g.transform(lambda x: x.rolling(6, min_periods=1).min())
        df[f"{col}_max6h"]  = g.transform(lambda x: x.rolling(6, min_periods=1).max())

    #std is nan when only 1 value in window fill with 0
    std_cols = [f"{c}_std6h" for c in rolling_cols]
    df[std_cols] = df[std_cols].fillna(0)

    # group 2 rate of change 
    delta_cols = ["HR", "O2Sat", "MAP", "Resp", "Lactate", "WBC", "SBP", "Temp"]

    for col in delta_cols:
        g = df.groupby("patient_id")[col]
        df[f"{col}_delta1h"] = df[col] - g.shift(1)
        df[f"{col}_delta3h"] = df[col] - g.shift(3)

    #beginning of stay has no prior values, fill with 0
    delta_feat_cols = [f"{c}_delta1h" for c in delta_cols] + [f"{c}_delta3h" for c in delta_cols]
    df[delta_feat_cols] = df[delta_feat_cols].fillna(0)

    #group 3 clinical composite features
    df["shock_index"]         = df["HR"] / (df["SBP"] + 1e-5)
    df["pulse_pressure"]      = df["SBP"] - df["DBP"]
    df["bun_creatinine_ratio"] = df["BUN"] / (df["Creatinine"] + 1e-5)
    df["resp_hr_ratio"]       = df["Resp"] / (df["HR"] + 1e-5)
    df["map_hr_product"]      = df["MAP"] * df["HR"]
    df["temp_deviation"]      = (df["Temp"] - 37.0).abs()

    #group 4 cumulative features since icu admission
    cum_cols = ["HR", "MAP", "WBC", "Lactate", "Resp"]

    for col in cum_cols:
        g = df.groupby("patient_id")[col]
        df[f"{col}_cummax"]  = g.transform("cummax")
        df[f"{col}_cummin"]  = g.transform("cummin")
        df[f"{col}_cummean"] = g.transform(lambda x: x.expanding(min_periods=1).mean())

    return df


 # apply to all three splits
print("engineering features for train")
df_train = engineer_features(df_train)
print("engineering features for val")
df_val = engineer_features(df_val)
print("engineering features for test")
df_test = engineer_features(df_test)


cols_after = [c for c in df_train.columns if c not in ["patient_id", "set", "SepsisLabel"]]
new_cols = [c for c in cols_after if c not in cols_before]

print(f"\nshape before: {shape_before}")
print(f"shape after:  {df_train.shape}")
print(f"new feature columns: {len(new_cols)}")
print(f"total features after engineering: {len(cols_after)} (was {len(cols_before)} before)")

print("\nnewly created columns:")
for i, c in enumerate(new_cols):
    print(f"  {i+1:3d}. {c}")

print("\nsample of new features:")
display(df_train[new_cols].head(10))

# verify no nans 
nan_counts = df_train[cols_after].isnull().sum()
nan_total = nan_counts.sum()
if nan_total > 0:
    print(f"\nwarning: {nan_total} nans found in these columns:")
    print(nan_counts[nan_counts > 0])
else:
    print("\nno nans in any feature column")

assert nan_total == 0, f"found {nan_total} nans in engineered features"


df_train.to_parquet(OUT_DIR + "train_featured.parquet", index=False)
df_val.to_parquet(OUT_DIR + "val_featured.parquet", index=False)
df_test.to_parquet(OUT_DIR + "test_featured.parquet", index=False)

# feature_names.json = all columns the model will use as input
feature_names = cols_after
with open(OUT_DIR + "feature_names.json", "w") as f:
    json.dump(feature_names, f, indent=2)

print(f"\nsaved train_featured.parquet ({len(df_train):,} rows)")
print(f"saved val_featured.parquet ({len(df_val):,} rows)")
print(f"saved test_featured.parquet ({len(df_test):,} rows)")
print(f"saved feature_names.json ({len(feature_names)} features)")



# count features per group
n_original = len(cols_before)
n_rolling = len([c for c in new_cols if any(c.endswith(s) for s in ["_mean6h", "_std6h", "_min6h", "_max6h"])])
n_delta = len([c for c in new_cols if "_delta" in c])
n_clinical = len(["shock_index", "pulse_pressure", "bun_creatinine_ratio",
                   "resp_hr_ratio", "map_hr_product", "temp_deviation"])
n_cumulative = len([c for c in new_cols if any(c.endswith(s) for s in ["_cummax", "_cummin", "_cummean"])])

groups = ["original\n(+indicators)", "rolling\nstats", "lag/delta", "clinical\ncomposites", "cumulative"]
counts = [n_original, n_rolling, n_delta, n_clinical, n_cumulative]
colors = ["#4c72b0", "#55a868", "#dd8452", "#c44e52", "#8172b3"]

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.bar(groups, counts, color=colors, edgecolor="black", linewidth=0.5)

for bar, val in zip(bars, counts):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
            str(val), ha="center", va="bottom", fontsize=12, fontweight="bold")

ax.set_ylabel("number of features")
ax.set_title(f"feature groups breakdown (total: {sum(counts)})", fontsize=13)
plt.tight_layout()
plt.savefig(OUT_DIR + "REPORT_feature_groups.png", dpi=150, bbox_inches="tight")
plt.show()


engineering features for train
engineering features for val
engineering features for test

shape before: (1060185, 52)
shape after:  (1060185, 129)
new feature columns: 77
total features after engineering: 126 (was 49 before)

newly created columns:
    1. HR_mean6h
    2. HR_std6h
    3. HR_min6h
    4. HR_max6h
    5. O2Sat_mean6h
    6. O2Sat_std6h
    7. O2Sat_min6h
    8. O2Sat_max6h
    9. MAP_mean6h
   10. MAP_std6h
   11. MAP_min6h
   12. MAP_max6h
   13. Resp_mean6h
   14. Resp_std6h
   15. Resp_min6h
   16. Resp_max6h
   17. Temp_mean6h
   18. Temp_std6h
   19. Temp_min6h
   20. Temp_max6h
   21. WBC_mean6h
   22. WBC_std6h
   23. WBC_min6h
   24. WBC_max6h
   25. Lactate_mean6h
   26. Lactate_std6h
   27. Lactate_min6h
   28. Lactate_max6h
   29. Creatinine_mean6h
   30. Creatinine_std6h
   31. Creatinine_min6h
   32. Creatinine_max6h
   33. SBP_mean6h
   34. SBP_std6h
   35. SBP_min6h
   36. SBP_max6h
   37. DBP_mean6h
   38. DBP_std6h
   39. DBP_min6h
   40. DBP_max6h
   4

,HR_mean6h,HR_std6h,HR_min6h,HR_max6h,O2Sat_mean6h,O2Sat_std6h,O2Sat_min6h,O2Sat_max6h,MAP_mean6h,MAP_std6h,...,MAP_cummean,WBC_cummax,WBC_cummin,WBC_cummean,Lactate_cummax,Lactate_cummin,Lactate_cummean,Resp_cummax,Resp_cummin,Resp_cummean
0,83.000000,0.000000,83.0,83.0,98.000000,0.000000,98.0,98.0,81.000000,0.000000,...,81.000000,10.3,10.3,10.3,1.59,1.59,1.59,18.0,18.0,18.000000
1,90.000000,9.899495,83.0,97.0,96.500000,2.121320,95.0,98.0,78.165000,4.009295,...,78.165000,10.3,10.3,10.3,1.59,1.59,1.59,19.0,18.0,18.500000
2,89.666667,7.023769,83.0,97.0,97.333333,2.081666,95.0,99.0,80.776667,5.338505,...,80.776667,10.3,10.3,10.3,1.59,1.59,1.59,22.0,18.0,19.666667
3,89.750000,5.737305,83.0,97.0,96.750000,2.061553,95.0,99.0,82.082500,5.081393,...,82.082500,10.3,10.3,10.3,1.59,1.59,1.59,30.0,18.0,22.250000
4,92.400000,7.733046,83.0,103.0,95.100000,4.098780,88.5,99.0,83.932000,6.038930,...,83.932000,10.3,10.3,10.3,1.59,1.59,1.59,30.0,18.0,22.700000
5,95.333333,9.973298,83.0,110.0,94.416667,4.030095,88.5,99.0,85.165000,6.188430,...,85.165000,10.3,10.3,10.3,1.59,1.59,1.59,30.0,18.0,22.583333
6,99.500000,8.961027,89.0,110.0,93.416667,3.693463,88.5,99.0,84.498333,6.901256,...,83.998571,10.3,10.3,10.3,1.59,1.59,1.59,30.0,18.0,23.500000
7,101.000000,9.208692,89.0,110.0,92.666667,3.763863,88.5,99.0,84.665000,6.642791,...,83.040000,10.3,10.3,10.3,1.59,1.59,1.59,30.0,18.0,24.187500
8,103.500000,7.092249,90.0,110.0,92.000000,2.588436,88.5,95.0,85.053333,6.802625,...,83.627778,10.3,10.3,10.3,1.59,1.59,1.59,30.0,18.0,24.388889
9,105.500000,3.082207,102.0,110.0,91.333333,2.136976,88.5,95.0,85.275000,6.861060,...,83.998000,10.3,10.3,10.3,1.59,1.59,1.59,30.0,18.0,24.950000



no nans in any feature column

saved train_featured.parquet (1,060,185 rows)
saved val_featured.parquet (228,969 rows)
saved test_featured.parquet (228,563 rows)
saved feature_names.json (126 features)


## 6.Label noise removal

In [7]:
# cell 6 - label noise mitigation

#sepsis labels are noisy physician records late

#three strategies I used to handle this
#label smoothing / softens hard 0/1 so model doesn't overfit noise
#transition window reweighting / downweight uncertain pre-onset rows
#confident learning / detect rows that are probably mislabeled

# %load_ext cudf.pandas
import pandas as pd
import numpy as np
import json
import lightgbm as lgb
from sklearn.model_selection import StratifiedKFold
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

OUT_DIR = "/kaggle/working/"
RANDOM_SEED = 42

try:
    _ = df_train.shape
except NameError:
    df_train = pd.read_parquet(OUT_DIR + "train_featured.parquet")
    print("loaded train_featured from parquet")

with open(OUT_DIR + "feature_names.json", "r") as f:
    feature_names = json.load(f)

# strategy 1 - label smoothing
# softens 1 -> 0.85 and 0 -> 0.05 so the model doesnt become
# overconfident on labels that might be wrong

df_train["smoothed_label"] = np.where(df_train["SepsisLabel"] == 1, 0.85, 0.05)

print("label smoothing applied")
print(f"  positive label value: 0.85, negative: 0.05")

# strategy 2 - transition window reweighting
# the 3 rows right before sepsis onset are labeled 0 but the patient
# is already deteriorating, so the label is uncertain. downweight them.
# positive rows get weight 2.0 to counter class imbalance.

df_train["sample_weight"] = 1.0

df_train.loc[df_train["SepsisLabel"] == 1, "sample_weight"] = 2.0

df_train = df_train.sort_values(["patient_id", "ICULOS"]).reset_index(drop=True)

sepsis_pids = df_train[df_train["SepsisLabel"] == 1]["patient_id"].unique()
transition_count = 0

for pid in sepsis_pids:
    mask = df_train["patient_id"] == pid
    pat_idx = df_train.loc[mask].index
    labels = df_train.loc[pat_idx, "SepsisLabel"]

    first_pos = labels[labels == 1].index[0]
    pos_in_pat = pat_idx.get_loc(first_pos) if hasattr(pat_idx, "get_loc") else list(pat_idx).index(first_pos)

    start = max(0, pos_in_pat - 3)
    transition_idx = pat_idx[start:pos_in_pat]

    df_train.loc[transition_idx, "sample_weight"] = 0.3
    transition_count += len(transition_idx)

print(f"  transition window rows downweighted: {transition_count}")

# strategy 3 - confident learning via quick lightgbm cross-val
# train a fast model and flag rows where prediction strongly
# disagrees with the label (likely mislabeled

X = df_train[feature_names].values
y = df_train["SepsisLabel"].values

oof_probs = np.zeros(len(y))
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_SEED)

print("\nrunning 5-fold cross-val for confident learning...")

for fold, (train_idx, val_idx) in enumerate(skf.split(X, y)):
    model = lgb.LGBMClassifier(
        n_estimators=100,
        n_jobs=-1,
        random_state=RANDOM_SEED,
        verbosity=-1
    )
    model.fit(X[train_idx], y[train_idx])
    oof_probs[val_idx] = model.predict_proba(X[val_idx])[:, 1]
    print(f"  fold {fold + 1}/5 done")

# flag suspected noise
#labeled positive but model is very confident its negative
noise_pos = (y == 1) & (oof_probs < 0.15)
#labeled negative but model is very confident its positive
noise_neg = (y == 0) & (oof_probs > 0.85)

suspected_noise = noise_pos | noise_neg
n_noise_pos = int(noise_pos.sum())
n_noise_neg = int(noise_neg.sum())
n_noise_total = int(suspected_noise.sum())

print(f"\nsuspected noisy rows detected: {n_noise_total} total ({n_noise_pos} in positives, {n_noise_neg} in negatives)")

# save the suspected noisy rows
df_noise = df_train.loc[suspected_noise, ["patient_id", "ICULOS", "SepsisLabel"]].copy()
df_noise["oof_prob"] = oof_probs[suspected_noise]
df_noise.to_csv(OUT_DIR + "suspected_noisy_rows.csv", index=False)
print(f"saved suspected_noisy_rows.csv ({len(df_noise)} rows)")

# display

print("\nsample of new columns:")
display(df_train[["patient_id", "ICULOS", "SepsisLabel", "smoothed_label", "sample_weight"]].head(30))

# save

df_train.to_parquet(OUT_DIR + "train_final.parquet", index=False)
print(f"\nsaved train_final.parquet ({len(df_train):,} rows)")

# report
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

# left: pie chart of sample_weight distribution
weight_counts = df_train["sample_weight"].value_counts().sort_index()
labels_pie = [f"weight={w}" for w in weight_counts.index]
colors_pie = ["#dd8452", "#4c72b0", "#55a868"]
ax1.pie(weight_counts.values, labels=labels_pie, autopct="%1.1f%%",
        colors=colors_pie[:len(weight_counts)], startangle=90, textprops={"fontsize": 10})
ax1.set_title("sample weight distribution", fontsize=12)

# right: bar chart of suspected noise
bars = ax2.bar(
    ["noisy positives\n(label=1, pred<0.15)", "noisy negatives\n(label=0, pred>0.85)"],
    [n_noise_pos, n_noise_neg],
    color=["#c44e52", "#4c72b0"], edgecolor="black", linewidth=0.5
)
for bar, val in zip(bars, [n_noise_pos, n_noise_neg]):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 10,
             f"{val:,}", ha="center", va="bottom", fontsize=11)
ax2.set_ylabel("number of rows")
ax2.set_title("suspected mislabeled rows", fontsize=12)

plt.suptitle("label noise mitigation summary", fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig(OUT_DIR + "REPORT_label_noise_summary.png", dpi=150, bbox_inches="tight")
plt.show()


label smoothing applied
  positive label value: 0.85, negative: 0.05
  transition window rows downweighted: 5078

running 5-fold cross-val for confident learning...
  fold 1/5 done
  fold 2/5 done
  fold 3/5 done
  fold 4/5 done
  fold 5/5 done

suspected noisy rows detected: 10722 total (10621 in positives, 101 in negatives)
saved suspected_noisy_rows.csv (10722 rows)

sample of new columns:


,patient_id,ICULOS,SepsisLabel,smoothed_label,sample_weight
0,p000001,1,0,0.05,1.0
1,p000001,2,0,0.05,1.0
2,p000001,3,0,0.05,1.0
3,p000001,4,0,0.05,1.0
4,p000001,5,0,0.05,1.0
5,p000001,6,0,0.05,1.0
6,p000001,7,0,0.05,1.0
7,p000001,8,0,0.05,1.0
8,p000001,9,0,0.05,1.0
9,p000001,10,0,0.05,1.0



saved train_final.parquet (1,060,185 rows)


## 7.Trying Logistic Regression

In [8]:
# cell 7 - baseline logistic regression

import pandas as pd
import numpy as np
import json
import joblib
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (roc_auc_score, average_precision_score, f1_score,
                             confusion_matrix, roc_curve, precision_recall_curve)
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns

OUT_DIR = "/kaggle/working/"
RANDOM_SEED = 42

# load data

df_train = pd.read_parquet(OUT_DIR + "train_final.parquet")
df_val   = pd.read_parquet(OUT_DIR + "val_featured.parquet")
df_test  = pd.read_parquet(OUT_DIR + "test_featured.parquet")

with open(OUT_DIR + "feature_names.json", "r") as f:
    feature_names = json.load(f)

X_train = df_train[feature_names]
X_val   = df_val[feature_names]
X_test  = df_test[feature_names]

y_train = df_train["SepsisLabel"].values
y_val   = df_val["SepsisLabel"].values
y_test  = df_test["SepsisLabel"].values

w_train = df_train["sample_weight"].values

print(f"features: {len(feature_names)}")
print(f"train: {len(y_train):,} rows  |  val: {len(y_val):,} rows  |  test: {len(y_test):,} rows")

# scale features - fit on train only

scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_val_sc   = scaler.transform(X_val)
X_test_sc  = scaler.transform(X_test)

joblib.dump(scaler, OUT_DIR + "scaler.pkl")
print("fitted scaler on train, saved scaler.pkl")

# train logistic regression

model = LogisticRegression(
    class_weight="balanced",
    max_iter=1000,
    random_state=RANDOM_SEED,
    n_jobs=-1
)
model.fit(X_train_sc, y_train, sample_weight=w_train)

joblib.dump(model, OUT_DIR + "baseline_model.pkl")
print("trained baseline model, saved baseline_model.pkl")

# evaluation helper

def evaluate(name, y_true, y_prob, threshold=0.5):
    y_pred = (y_prob >= threshold).astype(int)
    auroc = roc_auc_score(y_true, y_prob)
    auprc = average_precision_score(y_true, y_prob)
    f1 = f1_score(y_true, y_pred)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    sens = tp / (tp + fn) if (tp + fn) > 0 else 0
    spec = tn / (tn + fp) if (tn + fp) > 0 else 0
    metrics = {
        "auroc": round(auroc, 4),
        "auprc": round(auprc, 4),
        "f1": round(f1, 4),
        "sensitivity": round(sens, 4),
        "specificity": round(spec, 4),
        "tp": int(tp), "tn": int(tn), "fp": int(fp), "fn": int(fn)
    }
    return metrics

# predict probabilities
prob_val  = model.predict_proba(X_val_sc)[:, 1]
prob_test = model.predict_proba(X_test_sc)[:, 1]

metrics_val  = evaluate("val", y_val, prob_val)
metrics_test = evaluate("test", y_test, prob_test)

# print metrics side by side

print("\n" + "=" * 55)
print(f"{'metric':<16} {'validation':>16} {'test':>16}")
print("-" * 55)
for key in ["auroc", "auprc", "f1", "sensitivity", "specificity"]:
    print(f"  {key:<14} {metrics_val[key]:>16.4f} {metrics_test[key]:>16.4f}")
print("-" * 55)
for key in ["tp", "tn", "fp", "fn"]:
    print(f"  {key:<14} {metrics_val[key]:>16,} {metrics_test[key]:>16,}")
print("=" * 55)

# confusion matrix as dataframe
cm_df = pd.DataFrame({
    "val": {k: metrics_val[k] for k in ["tp", "tn", "fp", "fn"]},
    "test": {k: metrics_test[k] for k in ["tp", "tn", "fp", "fn"]}
})
display(cm_df)

# save all metrics
all_metrics = {"val": metrics_val, "test": metrics_test}
with open(OUT_DIR + "baseline_metrics.json", "w") as f:
    json.dump(all_metrics, f, indent=2)
print("\nsaved baseline_metrics.json")

# report 1 - confusion matrix heatmap/test set

cm = np.array([[metrics_test["tn"], metrics_test["fp"]],
               [metrics_test["fn"], metrics_test["tp"]]])

labels = np.array([
    [f"TN\n{metrics_test['tn']:,}", f"FP\n{metrics_test['fp']:,}"],
    [f"FN\n{metrics_test['fn']:,}", f"TP\n{metrics_test['tp']:,}"]
])

fig, ax = plt.subplots(figsize=(7, 6))
sns.heatmap(cm, annot=labels, fmt="", cmap="Blues", ax=ax,
            xticklabels=["predicted 0", "predicted 1"],
            yticklabels=["actual 0", "actual 1"],
            linewidths=1, cbar_kws={"shrink": 0.8})
ax.set_title("baseline logistic regression - confusion matrix (test)", fontsize=13)
ax.text(0.5, -0.12, f"sensitivity: {metrics_test['sensitivity']:.4f}    specificity: {metrics_test['specificity']:.4f}",
        transform=ax.transAxes, ha="center", fontsize=11)
plt.tight_layout()
plt.savefig(OUT_DIR + "REPORT_baseline_confusion_matrix.png", dpi=150, bbox_inches="tight")
plt.show()

# report 2 - roc curve /test set

fpr, tpr, _ = roc_curve(y_test, prob_test)

fig, ax = plt.subplots(figsize=(7, 6))
ax.plot(fpr, tpr, color="#4c72b0", linewidth=2, label=f"baseline LR (AUROC = {metrics_test['auroc']:.4f})")
ax.plot([0, 1], [0, 1], color="gray", linestyle="--", linewidth=1, label="random classifier")
ax.set_xlabel("false positive rate")
ax.set_ylabel("true positive rate")
ax.set_title("baseline logistic regression - roc curve (test)", fontsize=13)
ax.legend(loc="lower right")
ax.set_xlim([0, 1])
ax.set_ylim([0, 1.02])
plt.tight_layout()
plt.savefig(OUT_DIR + "REPORT_baseline_roc_curve.png", dpi=150, bbox_inches="tight")
plt.show()

# report 3 - precision-recall curve /test set

precision, recall, _ = precision_recall_curve(y_test, prob_test)
prevalence = y_test.mean()

fig, ax = plt.subplots(figsize=(7, 6))
ax.plot(recall, precision, color="#55a868", linewidth=2, label=f"baseline LR (AUPRC = {metrics_test['auprc']:.4f})")
ax.axhline(y=prevalence, color="gray", linestyle="--", linewidth=1, label=f"prevalence = {prevalence:.4f}")
ax.set_xlabel("recall")
ax.set_ylabel("precision")
ax.set_title("baseline logistic regression - precision-recall curve (test)", fontsize=13)
ax.legend(loc="upper right")
ax.set_xlim([0, 1])
ax.set_ylim([0, 1.02])
plt.tight_layout()
plt.savefig(OUT_DIR + "REPORT_baseline_pr_curve.png", dpi=150, bbox_inches="tight")
plt.show()


features: 126
train: 1,060,185 rows  |  val: 228,969 rows  |  test: 228,563 rows
fitted scaler on train, saved scaler.pkl
trained baseline model, saved baseline_model.pkl

metric                 validation             test
-------------------------------------------------------
  auroc                    0.7771           0.7777
  auprc                    0.0752           0.0871
  f1                       0.0562           0.0561
  sensitivity              0.8617           0.8596
  specificity              0.4740           0.4719
-------------------------------------------------------
  tp                        3,539            3,539
  tn                      106,575          105,914
  fp                      118,287          118,532
  fn                          568              578


,val,test
tp,3539,3539
tn,106575,105914
fp,118287,118532
fn,568,578



saved baseline_metrics.json


## 8.LightGBM

In [14]:
# cell 8 - lightgbm model

import pandas as pd
import numpy as np
import json
import joblib
import lightgbm as lgb
from sklearn.metrics import (roc_auc_score, average_precision_score, f1_score,
                             confusion_matrix, roc_curve, precision_recall_curve)
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns

OUT_DIR = "/kaggle/working/"
RANDOM_SEED = 42

# load data

df_train = pd.read_parquet(OUT_DIR + "train_final.parquet")
df_val   = pd.read_parquet(OUT_DIR + "val_featured.parquet")
df_test  = pd.read_parquet(OUT_DIR + "test_featured.parquet")

with open(OUT_DIR + "feature_names.json", "r") as f:
    feature_names = json.load(f)

X_train = df_train[feature_names]
y_train_smooth = df_train["smoothed_label"].values
y_train_hard   = df_train["SepsisLabel"].values
w_train        = df_train["sample_weight"].values

X_val  = df_val[feature_names]
y_val  = df_val["SepsisLabel"].values

X_test = df_test[feature_names]
y_test = df_test["SepsisLabel"].values

# load baseline metrics for comparison later
with open(OUT_DIR + "baseline_metrics.json", "r") as f:
    baseline_metrics = json.load(f)

print(f"features: {len(feature_names)}")
print(f"train: {len(y_train_hard):,}  |  val: {len(y_val):,}  |  test: {len(y_test):,}")

#1 temporal cross-validation for num_leaves tuning
# patients sorted by mean ICULOS, earlier patients train, later validate

# get mean iculos per patient to sort temporally
patient_order = df_train.groupby("patient_id")["ICULOS"].mean().sort_values()
sorted_pids = patient_order.index.tolist()
n_patients = len(sorted_pids)

num_leaves_options = [31, 63, 127]
n_folds = 5
cv_results = []

print("\ntemporal cv for num_leaves tuning...")
print(f"{'num_leaves':<12} {'fold':>6} {'auprc':>10}")
print("-" * 30)

for nl in num_leaves_options:
    fold_scores = []

    for fold in range(n_folds):
        # split patients temporally: first (fold+1)/6 for train, next chunk for val
        # each fold uses a progressively larger training set
        val_start = int(n_patients * (fold + 1) / (n_folds + 1))
        val_end   = int(n_patients * (fold + 2) / (n_folds + 1))

        train_pids = set(sorted_pids[:val_start])
        val_pids   = set(sorted_pids[val_start:val_end])

        tr_mask = df_train["patient_id"].isin(train_pids)
        vl_mask = df_train["patient_id"].isin(val_pids)

        # skip folds with too few positive samples
        if df_train.loc[vl_mask, "SepsisLabel"].sum() < 10:
            continue

        m = lgb.LGBMRegressor(
            objective="binary", n_estimators=300, learning_rate=0.05,
            num_leaves=nl, min_child_samples=50,
            subsample=0.8, colsample_bytree=0.8,
            n_jobs=-1, random_state=RANDOM_SEED, verbosity=-1
        )
        m.fit(
            df_train.loc[tr_mask, feature_names].values, df_train.loc[tr_mask, "smoothed_label"].values,
            sample_weight=df_train.loc[tr_mask, "sample_weight"].values,
            eval_set=[(df_train.loc[vl_mask, feature_names].values, df_train.loc[vl_mask, "SepsisLabel"].values)],
            callbacks=[lgb.early_stopping(30, verbose=False)]
        )
        preds = np.clip(m.predict(df_train.loc[vl_mask, feature_names].values), 0, 1)
        score = average_precision_score(df_train.loc[vl_mask, "SepsisLabel"], preds)
        fold_scores.append(score)
        print(f"  {nl:<12} {fold+1:>4}   {score:>10.4f}")

    mean_score = np.mean(fold_scores)
    std_score  = np.std(fold_scores)
    cv_results.append({"num_leaves": nl, "mean_auprc": mean_score,
                        "std_auprc": std_score, "fold_scores": fold_scores})

#pick best
best_cv = max(cv_results, key=lambda x: x["mean_auprc"])
best_num_leaves = best_cv["num_leaves"]

print(f"\ncv summary:")
print(f"{'num_leaves':<12} {'mean_auprc':>12} {'std':>10}")
print("-" * 36)
for r in cv_results:
    marker = " <-- best" if r["num_leaves"] == best_num_leaves else ""
    print(f"  {r['num_leaves']:<10} {r['mean_auprc']:>12.4f} {r['std_auprc']:>10.4f}{marker}")

print(f"\nbest num_leaves: {best_num_leaves} | best auprc: {best_cv['mean_auprc']:.4f}")

#2 final lgbm training

print("\ntraining final lightgbm model...")

final_model = lgb.LGBMRegressor(
    objective="binary", metric="average_precision",
    n_estimators=1000, learning_rate=0.05,
    num_leaves=best_num_leaves, min_child_samples=50,
    subsample=0.8, colsample_bytree=0.8,
    n_jobs=-1, random_state=RANDOM_SEED, verbosity=-1
)

final_model.fit(
    X_train.values, y_train_smooth,
    sample_weight=w_train,
    eval_set=[(X_val.values, y_val)],
    callbacks=[lgb.early_stopping(50, verbose=True), lgb.log_evaluation(100)]
)

print(f"best iteration: {final_model.best_iteration_}")

#3 threshold selection on validation set

prob_val  = np.clip(final_model.predict(X_val.values), 0, 1)
prob_test = np.clip(final_model.predict(X_test.values), 0, 1)

# precision-recall on val to find thresholds
prec_v, rec_v, thresholds_v = precision_recall_curve(y_val, prob_val)

# threshold_f1: maximizes f1
# f1 = 2*prec*rec/(prec+rec), thresholds array is 1 shorter than prec/rec
f1_scores = 2 * prec_v[:-1] * rec_v[:-1] / (prec_v[:-1] + rec_v[:-1] + 1e-10)
best_f1_idx = np.argmax(f1_scores)
threshold_f1 = float(thresholds_v[best_f1_idx])

# threshold_sens: lowest threshold where sensitivity >= 0.80
# sensitivity = recall, and recall decreases as threshold increases
# so we want the highest threshold where recall >= 0.80
sens_mask = rec_v[:-1] >= 0.80
if sens_mask.any():
    # highest threshold (rightmost) where recall is still >= 0.80
    threshold_sens = float(thresholds_v[np.where(sens_mask)[0][-1]])
else:
    threshold_sens = 0.5

print(f"\nbest f1 threshold: {threshold_f1:.4f} (f1 = {f1_scores[best_f1_idx]:.4f})")
print(f"sensitivity-80% threshold: {threshold_sens:.4f}")

thresholds_dict = {"threshold_f1": threshold_f1, "threshold_sens": threshold_sens}
with open(OUT_DIR + "lgbm_thresholds.json", "w") as f:
    json.dump(thresholds_dict, f, indent=2)

# part 4 - evaluation on test set at threshold_f1

def evaluate(y_true, y_prob, threshold):
    y_pred = (y_prob >= threshold).astype(int)
    auroc = roc_auc_score(y_true, y_prob)
    auprc = average_precision_score(y_true, y_prob)
    f1 = f1_score(y_true, y_pred)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    sens = tp / (tp + fn) if (tp + fn) > 0 else 0
    spec = tn / (tn + fp) if (tn + fp) > 0 else 0
    return {
        "auroc": round(auroc, 4), "auprc": round(auprc, 4), "f1": round(f1, 4),
        "sensitivity": round(sens, 4), "specificity": round(spec, 4),
        "tp": int(tp), "tn": int(tn), "fp": int(fp), "fn": int(fn),
        "threshold": round(threshold, 4)
    }

lgbm_val  = evaluate(y_val, prob_val, threshold_f1)
lgbm_test = evaluate(y_test, prob_test, threshold_f1)

# side-by-side comparison with baseline
print("\n" + "=" * 65)
print(f"{'metric':<16} {'baseline (test)':>18} {'lgbm (test)':>18}")
print("-" * 65)
for key in ["auroc", "auprc", "f1", "sensitivity", "specificity"]:
    bv = baseline_metrics["test"][key]
    lv = lgbm_test[key]
    diff = lv - bv
    arrow = "+" if diff > 0 else ""
    print(f"  {key:<14} {bv:>18.4f} {lv:>14.4f} ({arrow}{diff:.4f})")
print("-" * 65)
for key in ["tp", "tn", "fp", "fn"]:
    print(f"  {key:<14} {baseline_metrics['test'][key]:>18,} {lgbm_test[key]:>18,}")
print("=" * 65)

# save
lgbm_metrics = {"val": lgbm_val, "test": lgbm_test}
with open(OUT_DIR + "lgbm_metrics.json", "w") as f:
    json.dump(lgbm_metrics, f, indent=2)

joblib.dump(final_model, OUT_DIR + "lgbm_model.pkl")
print("\nsaved lgbm_model.pkl, lgbm_metrics.json, lgbm_thresholds.json")

# report 1 - temporal cv results

fig, ax = plt.subplots(figsize=(8, 5))
nl_vals   = [r["num_leaves"] for r in cv_results]
means     = [r["mean_auprc"] for r in cv_results]
stds      = [r["std_auprc"] for r in cv_results]

ax.errorbar(nl_vals, means, yerr=stds, marker="o", capsize=5,
            color="#4c72b0", linewidth=2, markersize=8)
ax.axvline(best_num_leaves, color="red", linestyle="--", alpha=0.5, label=f"best = {best_num_leaves}")
ax.set_xlabel("num_leaves")
ax.set_ylabel("mean auprc")
ax.set_title("temporal cv: num_leaves tuning", fontsize=13)
ax.set_xticks(nl_vals)
ax.legend()
plt.tight_layout()
plt.savefig(OUT_DIR + "REPORT_lgbm_cv_results.png", dpi=150, bbox_inches="tight")
plt.show()

# report 2 - feature importance (top 25 by gain)

importance = final_model.booster_.feature_importance(importance_type="gain")
feat_imp = pd.DataFrame({"feature": feature_names, "importance": importance})
feat_imp = feat_imp.sort_values("importance", ascending=True).tail(25)

# color-code by feature group
def get_color(name):
    if any(name.endswith(s) for s in ["_mean6h", "_std6h", "_min6h", "_max6h"]):
        return "#dd8452"   # orange - rolling
    if "_delta" in name:
        return "#55a868"   # green - delta/lag
    if name in ["shock_index", "pulse_pressure", "bun_creatinine_ratio",
                "resp_hr_ratio", "map_hr_product", "temp_deviation"]:
        return "#c44e52"   # red - clinical composite
    if any(name.endswith(s) for s in ["_cummax", "_cummin", "_cummean"]):
        return "#8172b3"   # purple - cumulative
    return "#4c72b0"       # blue - original

colors = [get_color(f) for f in feat_imp["feature"]]

fig, ax = plt.subplots(figsize=(10, 12))
ax.barh(feat_imp["feature"], feat_imp["importance"], color=colors, edgecolor="black", linewidth=0.3)
ax.set_xlabel("importance (gain)")
ax.set_title("lgbm top 25 features by gain", fontsize=13)

# legend for color groups
from matplotlib.patches import Patch
legend_items = [
    Patch(facecolor="#4c72b0", label="original"),
    Patch(facecolor="#dd8452", label="rolling stats"),
    Patch(facecolor="#55a868", label="delta/lag"),
    Patch(facecolor="#c44e52", label="clinical composite"),
    Patch(facecolor="#8172b3", label="cumulative"),
]
ax.legend(handles=legend_items, loc="lower right", fontsize=9)
plt.tight_layout()
plt.savefig(OUT_DIR + "REPORT_lgbm_feature_importance.png", dpi=150, bbox_inches="tight")
plt.show()

# report 3 - pr curve with both thresholds marked

prec_t, rec_t, thr_t = precision_recall_curve(y_test, prob_test)
prevalence = y_test.mean()

# find closest points on curve for each threshold
def find_pr_at_threshold(prec, rec, thr, target_thr):
    idx = np.argmin(np.abs(thr - target_thr))
    return rec[idx], prec[idx]

r_f1, p_f1 = find_pr_at_threshold(prec_t, rec_t, thr_t, threshold_f1)
r_s, p_s   = find_pr_at_threshold(prec_t, rec_t, thr_t, threshold_sens)

fig, ax = plt.subplots(figsize=(9, 6))
ax.plot(rec_t, prec_t, color="#55a868", linewidth=2, label=f"lgbm (auprc = {lgbm_test['auprc']:.4f})")
ax.axhline(y=prevalence, color="gray", linestyle="--", linewidth=1, label=f"prevalence = {prevalence:.4f}")

ax.scatter([r_f1], [p_f1], color="#4c72b0", s=120, zorder=5, edgecolors="black")
ax.annotate(f"f1-optimal\nthr={threshold_f1:.3f}", (r_f1, p_f1),
            textcoords="offset points", xytext=(15, 10), fontsize=9,
            arrowprops=dict(arrowstyle="->", color="#4c72b0"), color="#4c72b0")

ax.scatter([r_s], [p_s], color="#c44e52", s=120, zorder=5, edgecolors="black")
ax.annotate(f"sens>=0.80\nthr={threshold_sens:.3f}", (r_s, p_s),
            textcoords="offset points", xytext=(15, -20), fontsize=9,
            arrowprops=dict(arrowstyle="->", color="#c44e52"), color="#c44e52")

ax.set_xlabel("recall")
ax.set_ylabel("precision")
ax.set_title("lgbm precision-recall curve with threshold selection (test)", fontsize=13)
ax.legend(loc="upper right")
ax.set_xlim([0, 1])
ax.set_ylim([0, 1.02])
plt.tight_layout()
plt.savefig(OUT_DIR + "REPORT_lgbm_pr_curve_thresholds.png", dpi=150, bbox_inches="tight")
plt.show()

# report 4 - confusion matrix heatmap (test, threshold_f1)

cm = np.array([[lgbm_test["tn"], lgbm_test["fp"]],
               [lgbm_test["fn"], lgbm_test["tp"]]])
labels_cm = np.array([
    [f"TN\n{lgbm_test['tn']:,}", f"FP\n{lgbm_test['fp']:,}"],
    [f"FN\n{lgbm_test['fn']:,}", f"TP\n{lgbm_test['tp']:,}"]
])

fig, ax = plt.subplots(figsize=(7, 6))
sns.heatmap(cm, annot=labels_cm, fmt="", cmap="Greens", ax=ax,
            xticklabels=["predicted 0", "predicted 1"],
            yticklabels=["actual 0", "actual 1"],
            linewidths=1, cbar_kws={"shrink": 0.8})
ax.set_title(f"lgbm confusion matrix (test, thr={threshold_f1:.3f})", fontsize=13)
ax.text(0.5, -0.12,
        f"sensitivity: {lgbm_test['sensitivity']:.4f}    specificity: {lgbm_test['specificity']:.4f}",
        transform=ax.transAxes, ha="center", fontsize=11)
plt.tight_layout()
plt.savefig(OUT_DIR + "REPORT_lgbm_confusion_matrix.png", dpi=150, bbox_inches="tight")
plt.show()

# report 5 - roc curve: lgbm vs baseline

# need baseline probs on test - reload baseline model and scaler
baseline_model = joblib.load(OUT_DIR + "baseline_model.pkl")
scaler = joblib.load(OUT_DIR + "scaler.pkl")
X_test_sc = scaler.transform(X_test)
prob_test_baseline = baseline_model.predict_proba(X_test_sc)[:, 1]

fpr_b, tpr_b, _ = roc_curve(y_test, prob_test_baseline)
fpr_l, tpr_l, _ = roc_curve(y_test, prob_test)

fig, ax = plt.subplots(figsize=(8, 6))
ax.plot(fpr_b, tpr_b, color="#dd8452", linewidth=2,
        label=f"baseline LR (auroc = {baseline_metrics['test']['auroc']:.4f})")
ax.plot(fpr_l, tpr_l, color="#4c72b0", linewidth=2,
        label=f"lgbm (auroc = {lgbm_test['auroc']:.4f})")
ax.plot([0, 1], [0, 1], color="gray", linestyle="--", linewidth=1, label="random")
ax.set_xlabel("false positive rate")
ax.set_ylabel("true positive rate")
ax.set_title("roc curve comparison: lgbm vs baseline (test)", fontsize=13)
ax.legend(loc="lower right")
ax.set_xlim([0, 1])
ax.set_ylim([0, 1.02])
plt.tight_layout()
plt.savefig(OUT_DIR + "REPORT_lgbm_vs_baseline_roc.png", dpi=150, bbox_inches="tight")
plt.show()


features: 126
train: 1,060,185  |  val: 228,969  |  test: 228,563

temporal cv for num_leaves tuning...
num_leaves     fold      auprc
------------------------------
  31              1       0.0186
  31              2       0.0101
  31              3       0.0041
  31              4       0.0048
  31              5       0.0250
  63              1       0.0186
  63              2       0.0101
  63              3       0.0041
  63              4       0.0048
  63              5       0.0250
  127             1       0.0186
  127             2       0.0101
  127             3       0.0041
  127             4       0.0048
  127             5       0.0250

cv summary:
num_leaves     mean_auprc        std
------------------------------------
  31               0.0125     0.0081 <-- best
  63               0.0125     0.0081
  127              0.0125     0.0081

best num_leaves: 31 | best auprc: 0.0125

training final lightgbm model...
Training until validation scores don't improve for 50 ro